# Section 0 — start here

**Synthetic onboarding, not empirical analysis.** Section 0 provides one shared configuration, explicit session calendar, publication-aware observations, eligibility/anchor conventions, benchmark interpolation, baseline contract path, canonical holdings and exact-contract accounting.

The interpolation and state-machine examples are separate from the hand-authored accounting holdings. They do not constitute the full historical portfolio pipeline. Phase B production data, sourced first-notice coverage, real session timing, benchmark/baseline sizing, sample gates and AUM preflight remain outstanding. Never silently replace a delivery-avoidance anchor with expiration.

Run from a fresh Python kernel in this repository or this notebook's folder, using the project's uv environment. All code is offline and fixture-only; saved outputs are cleared. Each teammate still owns their Part 2 engine, tests and issue-linked PR, and must review teammate PRs. This onboarding contribution does not fulfil their implementation task.

### Environment and kernel: use the existing lockfile

Before opening a notebook, run these commands in the current checkout's repository root:

```bash
uv sync --locked
uv run --no-sync python -c "import sys, ipykernel, nbformat; print(sys.executable)"
```

The first command synchronises the existing shared lockfile; it may need network access to obtain packages. The second verifies the declared notebook packages and prints the interpreter to select. Notebook examples themselves run offline after setup. Do not independently add packages, use `uv add` or `pip install`, or edit `pyproject.toml`/`uv.lock`. If synchronisation or imports fail, report the error to Jacopo rather than changing dependencies.

In VS Code, use **Select Kernel → Python Environments** and select the interpreter printed above for this checkout (normally `.venv/bin/python`). With an existing Jupyter frontend, select a kernel backed by that same interpreter. If it is not listed, coordinate kernel setup with Jacopo; do not substitute an unrelated environment. Restart the kernel, start it in this repository or the notebook's folder, then run all cells. A separate worktree must use its intended project environment; do not reuse a kernel already importing another checkout.

In [ ]:
from pathlib import Path
from decimal import Decimal
from pprint import pprint
import sys

# Start the kernel in this repository or the notebook's own folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "config/project.yaml").is_file()
             and (p / "data/fixtures/phase_a.json").is_file()
             and (p / "src/__init__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the kernel in the repository or this notebook's folder.")
sys.path.insert(0, str(ROOT))
import src
assert Path(src.__file__).resolve().parent == ROOT / "src", "Restart the kernel for this checkout."
from src.validation.schemas import load_project_config, HOLDINGS_COLUMNS
from src.data.fixtures import load_fixture

config = load_project_config(ROOT / "config/project.yaml")
f = load_fixture(ROOT / "data/fixtures/phase_a.json",
                 config["max_liquidity_source_age_sessions"])
return_date = f.raw["return_dates"][0]
context = f.calendar.decision(return_date)
print("SYNTHETIC FIXTURES ONLY", config["roots"], "return interval ends", return_date)


### Locked portfolio and shared configuration

Corn (ZC), Soybeans (ZS) and Chicago SRW Wheat (ZW) share approximately equal target notionals. Read the configuration; do not independently redefine weights, horizons, participation, metric or reference values. The fixture's $1m denominator is an arithmetic example, not evidence that any empirical capacity preflight passed.

In [ ]:
keys = ["project_spec_version", "roots", "target_weights", "benchmark_target_days",
        "baseline_roll_offset_trading_days", "capacity_participation_rate",
        "capacity_liquidation_horizon_days", "adv_lookback_days",
        "tracking_min_common_dates", "tracking_min_candidate_survival",
        "rounding_reference_aum_usd", "selection_roll_reference_aum_usd",
        "capacity_reference_aum_usd", "max_liquidity_source_age_sessions"]
pprint({key: config[key] for key in keys})


### Explicit sessions and decision cut-offs

A position for `(t-1,t]` is fixed at the preceding session's cut-off. Each session has explicit times; no generic weekday approximation or universal closing time is inferred. The fixture contains a non-session and a shortened-session example solely to exercise the interface.

In [ ]:
pprint(context.__dict__)
last_context = f.calendar.decision(f.raw["return_dates"][-1])
pprint(f.calendar.sessions[last_context.decision_date].__dict__)
assert f.raw["synthetic_non_session"] not in f.calendar.indices


### Observation availability

Trading dates come from `ts_ref` during normalisation; publication timestamps are retained separately. Select revisions known by the decision cut-off, not by a one-date shift alone. A later valuation snapshot may supply settlements for P&L but must never feed a past position decision.

In [ ]:
selection = f.observations.latest_available("ZC_SYN_NEAR", "cleared_volume", context)
assert selection.observation is not None
view = selection.observation.view(context, selected=True)
pprint({k: view[k] for k in ("contract", "trading_date", "publication_timestamp",
                            "decision_cutoff", "available_as_of", "lag_from_decision_sessions")})
late_revision = next(r for r in f.observations.observations if r.statistic_value == 999999)
assert not late_revision.view(context)["available_as_of"]
print("Late revision excluded at this cut-off.")


### Strictly lagged ADV20

The decision session never enters ADV20, even if its volume has already been published. Use exactly 20 consecutive verified prior sessions with publication availability, no skipped interior observations and the configured endpoint freshness bound. Non-sessions are excluded.

In [ ]:
adv = f.observations.adv20("ZC_SYN_NEAR", context)
assert adv["available"] and len(adv["rows"]) == 20
assert all(r["session_index"] < context.decision_index for r in adv["rows"])
assert all(r["trading_date"] != f.raw["synthetic_non_session"] for r in adv["rows"])
pprint({k: adv[k] for k in ("window_start", "window_end", "observations",
                           "endpoint_lag_from_decision_sessions", "adv20_contracts")})


### Shared 60-observed-session interpolation

These are not destination alternatives. The unconstrained benchmark brackets 60 sessions to the explicit delivery-avoidance anchor; if unbracketed it uses the nearest eligible maturity with a fallback flag. The fixture's anchors are synthetic, not sourced first-notice dates.

In [ ]:
from src.benchmark.constant_maturity import constant_maturity_weights
curve = [r for r in f.raw["eligible_curve"] if r["commodity"] == "ZC" and r["eligible"]]
weights = constant_maturity_weights({r["contract"]: r["days_to_anchor"] for r in curve},
                                    config["benchmark_target_days"])
assert sum(weights.weights.values()) == 1
pprint(weights.__dict__)
unbracketed = f.raw["interpolation_cases"][-1]
pprint(constant_maturity_weights(unbracketed["days"], config["benchmark_target_days"]).__dict__)


### Generated baseline path

The shared state machine starts nearest eligible, holds until its deadline, and rolls to the next deferred eligible contract five observed sessions before the outgoing anchor. Event dates are decision dates; generated positions apply to the next return interval. This demonstration selects contract identities only, not capacity-sized quantities.

**Complete-curve precondition:** `build_baseline_path` assumes a verified complete eligible curve for each decision date, using only information available by that cut-off. Absence of the active contract from a supplied complete curve may represent confirmed loss of eligibility. An unexplained missing contract row is not equivalent to confirmed ineligibility and must not be passed through as an eligibility decision. Production data preparation must validate curve completeness before calling the state machine; the helper cannot distinguish a missing-data omission from a confirmed removal. The explicit synthetic baseline case is supplied as a complete hand fixture. No state-machine behaviour is changed here.

In [ ]:
from src.portfolio.baseline import build_baseline_path
case = f.raw["baseline_path_case"]
path = build_baseline_path(case["eligible_curve"], case["decision_dates"],
                           case["commodities"], f.calendar, f.catalogue)
event = path.roll_events[0]
pprint(event)
pprint(path.positions[:3])


### Canonical holdings and exact-contract settlement changes

The accounting fixture deliberately supplies fixed benchmark, identical and half-sized quantities. It is not a historical benchmark or full baseline construction. `Holding` records decision provenance; `HoldingsBook` validates identities and supplied integer ceilings. The same exact contract supplies both consecutive settlement endpoints.

In [ ]:
row = next(r for r in f.holdings.rows if r.date == return_date and r.strategy == "benchmark")
pprint({k: row.to_record()[k] for k in ("date", "strategy", "commodity", "contract",
                                      "benchmark_fractional_contracts", "decision_cutoff",
                                      "formed_at", "inputs_as_of")})
marks = f.observations.return_inputs({"benchmark": {row.contract: row.quantity}}, return_date,
                                     f.raw["snapshot_end_exclusive"])
old, new = marks["price_endpoints"][row.contract]
assert old.observation.identity == new.observation.identity
change = new.observation.statistic_value - old.observation.statistic_value
leg_pnl = row.quantity * f.catalogue[row.contract].dollars_per_price_unit * change
pprint({"exact_contract": row.contract, "settlement_change": change, "leg_pnl_usd": leg_pnl})


### Shared P&L, fixed-AUM returns and Tracking RMSE

P&L uses pre-existing quantities × verified dollars per price unit × exact-contract settlement change. Both benchmark and implementations divide by the same fixed AUM. The engine uses one common valid-date mask and invalidates dates with missing required settlements; it never zero-fills or splices maturities. The later daily ranking metric is `10000 × sqrt(mean((implementation_return - benchmark_return)**2))`, without annualisation. The checks below verify synthetic arithmetic only.

In [ ]:
from src.metrics.portfolio_pnl import portfolio_pnl
from src.metrics.tracking_rmse_bps import tracking_rmse_bps
daily = portfolio_pnl(f.holdings, f.observations, f.raw["return_dates"], f.raw["strategies"],
                      f.raw["fixed_aum"], f.raw["snapshot_end_exclusive"])
pprint([{"date": r.date, "common_valid": r.common_valid, "pnl_usd": r.pnl,
         "daily_return": r.returns, "fixed_aum": r.fixed_aum} for r in daily[:1]])
scores = tracking_rmse_bps(daily)
assert scores["fixture_baseline"] == 0 and scores["half_size_fixture"] > 0
pprint({"synthetic_tracking_rmse_bps": scores})


### Handoff: four independent strands

| Teammate / subject | Consume from Section 0 | Implement independently |
| --- | --- | --- |
| Varshini / destination | Eligible curves, anchors, baseline roll events and benchmark | Destination engine and hand tests; keep timing, capacity and rounding fixed. [Issue #12](https://github.com/JacopoMichelacci/Futures_Class/issues/12). |
| Lucie / timing | Same outgoing/incoming pair, explicit sessions, publication-aware volume/OI | Timing engine and hand tests; keep destination, capacity and rounding fixed. The Part 2 engine issue was not present in the live roadmap checked on 11 October; create/confirm it before the graded PR. |
| Jacopo / capacity | Lagged ADV20, explicit contract metadata, target quantities and deferred ordering | Capacity-handling engine and hand tests; keep destination, timing and rounding fixed. [Issue #13](https://github.com/JacopoMichelacci/Futures_Class/issues/13). |
| Alan / rounding | Same exact contracts, post-capacity fractional targets and integer ceilings | Rounding engine and hand tests; keep destination, timing and capacity fixed. [Issue #6](https://github.com/JacopoMichelacci/Futures_Class/issues/6). |

All four emit canonical holdings and use the same benchmark, `portfolio_pnl`, fixed AUM and `tracking_rmse_bps` on common valid dates. No strand consumes another strand's chosen method. Integration and empirical method selection are later work.

See [shared interfaces](../docs/section0_phase_a.md) and [Phase A handoff](../outputs/section0/HANDOFF.md). From the repository root: `uv run --no-sync python -B -m unittest discover -s tests/unit -v` and `uv run --no-sync python -B scripts/run_section0.py --fixtures`. These notebooks are a separate onboarding contribution built on the merged Section 0 infrastructure in [PR #18](https://github.com/JacopoMichelacci/Futures_Class/pull/18).